# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print overview from dataset.metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns and their `@id`s.

In [ ]:
# List all record sets by @id
print("Record sets (by @id):")
for record_set in dataset.metadata.record_sets:
    print(f"  - {record_set['@id']}: {record_set.get('name','')} ({record_set.get('description','')})")

# For each record set, list its fields and columns by @id
for record_set in dataset.metadata.record_sets:
    print(f"\nRecord Set @id: {record_set['@id']}")
    # Fields
    if 'field' in record_set:
        print("  Fields:")
        for field in record_set['field']:
            print(f"    - {field['@id']} | name: {field.get('name','')} | description: {field.get('description','')}")
    # Columns
    if 'column' in record_set:
        print("  Columns:")
        for column in record_set['column']:
            print(f"    - {column['@id']} | name: {column.get('name','')} | dataType: {column.get('dataType','')}")

# Choose the first record set for demonstration
if len(dataset.metadata.record_sets) > 0:
    example_record_set = dataset.metadata.record_sets[0]['@id']
    print(f"\nExample Record Set selected for data extraction: {example_record_set}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# List all record sets' @id
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]

# Prepare a DataFrame for each record set
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns for the first record set
if record_sets:
    print(f"Columns in record set {record_sets[0]}:")
    print(dataframes[record_sets[0]].columns.tolist())
    display(dataframes[record_sets[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping. This example focuses on the first record set for sample analysis.

In [ ]:
# Replace these with actual field (column) @ids from the dataset -- update as suitable based on Section 2 output
record_set_id = record_sets[0]  # Demo: use the first
df = dataframes[record_set_id]

# List numeric columns to choose one
numeric_cols = df.select_dtypes(include='number').columns.tolist()
print(f"Numeric columns: {numeric_cols}")

if numeric_cols:
    numeric_field = numeric_cols[0]  # Select first numeric column for demonstration

    # Filter entries where numeric_field > threshold
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records in {record_set_id} with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a possible categorical column
    candidate_groups = df.select_dtypes(include='object').columns.tolist()
    if candidate_groups:
        group_field = candidate_groups[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data (mean {numeric_field}) by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields. Here, we plot the distribution of a numeric field and a grouped bar if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_cols:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Group barplot if group_field exists and is categorical with not too many categories
    if 'group_field' in locals():
        top_groups = df[group_field].value_counts().index[:5]
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_groups)])
        plt.title(f"Mean {numeric_field} by {group_field} (top 5 groups)")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, review, and process the FAIR² mass spectrometry dataset using mlcroissant.

- We listed the record sets, fields, and columns by their `@id` to ensure reproducibility.
- We explored numeric fields, filtered, normalized, and visualized distributions.

For deeper analysis, refer to the column and record set `@id` values revealed above, or use mlcroissant utilities to further explore relationships in the FAIR² dataset.